# 03 — Modeling

Walk-forward validation of the volatility **regressor** (forward realized volatility) and the volatility **regime classifier** (calm vs. elevated), then inspect feature importance.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from coinpredictor.data.ohlcv import load_ohlcv
from coinpredictor.features import build_features, split_xy
from coinpredictor.model import (
    build_baseline, build_vol_regressor, build_regime_classifier,
    walk_forward_regress, walk_forward_validate,
)

feats = build_features(load_ohlcv())
X, y_vol = split_xy(feats)
_, y_regime = split_xy(feats, regime=True)
X.shape, y_vol.mean(), y_regime.mean()

In [ ]:
# Regression: forward realized volatility
print(walk_forward_regress(build_baseline(), X, y_vol, model_name='Ridge baseline').summary())
print(walk_forward_regress(build_vol_regressor(), X, y_vol, model_name='LightGBM').summary())

# Classification: high-volatility regime (much more predictable than direction)
print(walk_forward_validate(build_regime_classifier(), X, y_regime, model_name='Regime LGBM').summary())

> The regime classifier typically reaches **~60–65% accuracy / AUC ~0.64** — meaningfully better than the ~51% achievable for price direction, because volatility *clusters*. Regression R² on the vol level is often modest (a few extreme spikes dominate the error), but the correlation and regime signal are what drive the strategy.

In [ ]:
# Fit final volatility regressor and view feature importances
import pandas as pd
from coinpredictor.model import regressor_importances
model = build_vol_regressor().fit(X, y_vol)
imp = pd.Series(regressor_importances(model), index=X.columns).sort_values()
imp.tail(15).plot.barh(figsize=(7, 6), title='Top volatility-regressor features')

In [ ]:
# Persist the trained artifacts for the dashboard / predict CLI
from coinpredictor.model import train_and_save
art = train_and_save(feats)
print(art.reg_cv.summary())
print(art.clf_cv.summary() if art.clf_cv else 'no classifier')